In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pace_pic_validation.io.calcification_pic_processing import pic_from_ca

## i need volumes of filtrations to finish this


In [ ]:

BE20260729_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Research_Campaigns/Norbjørn /August 2026/data/BE20260729_salinity.csv'
SS20260601_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj2_CalcificationRates_of_Coccoblooms/Data/SS20260601_Incubations_no_RM.csv'


BE20260729_df = pd.read_csv(BE20260729_path)
SS20260601_df = pd.read_csv(SS20260601_path)

print(BE20260729_df.columns)


print(SS20260601_df['Sample ID'])



In [ ]:
SS20260601_df = SS20260601_df.rename(columns={'Sample ID': 'sample_id'})
SS20260601_df['date_measured_alk'] = 'NA'

parts = SS20260601_df["sample_id"].astype("string").str.extract(
    r"^(?P<date>\d{8})_(?P<station>\d+)(?P<sample>[A-D]+)_(?P<time>t[01])$"
)

parsed_ids = pd.DataFrame({
    "Cruise": "SS" + parts["date"],
    "Station": pd.to_numeric(parts["station"], errors="coerce").astype("Int64"),
    "Sample ID": parts["sample"],
    "Time ID": parts["time"],
}, index=SS20260601_df.index)


SS20260601_df = pd.concat([SS20260601_df, parsed_ids], axis=1)





SS20260601_df = SS20260601_df.drop(columns=['Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30',
       'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34',
       'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38',
       'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42',
       'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46',
       'Unnamed: 47', 'r2_1', 'r2_2', 'r2_3', 'r2_4','Sal1 (K*2)', 'Sal2 ( (K*2)', 'Sal3  (K*2)',
       'Sal4   (K*2)', 'Sal5  (K*2)', 'SAL_final (two best / AVG)','points_1',
       'points_2', 'points_3', 'points_4','m_1', 'm_2', 'm_3', 'm_4', 'Unnamed: 24','sample_id', 'time ID'])

BE20260729_df = BE20260729_df.drop(columns=['SAL_1 ', 'SAL_2 ',
       'SAL_3 ', 'SAL_4 ', 'SAL_5', 'SAL_mean (K)', 'm_1', 'R_1','m_2', 'R_2','m_3', 'R_3', 'm_4', 'R_4','PIC (mg/L)', 'Unnamed: 25',])


SS20260601_df = SS20260601_df.rename(columns={'Alk_1  (µmol/kg)':'Alk_1 (µmol/kg)',
                                              'Alk_2  (µmol/kg)':'Alk_2 (µmol/kg)',
                                              'Alk_3  (µmol/kg)':'Alk_3 (µmol/kg)',
                                              'Alk_4':'Alk_4 (µmol/kg)',
                                              'Time ID ': 'Time ID'
                                              })

BE20260729_df = BE20260729_df.rename(columns={'Alk_1 ':'Alk_1 (µmol/kg)',
                                              'Alk_2 ':'Alk_2 (µmol/kg)',
                                              'Alk_3':'Alk_3 (µmol/kg)',
                                              'Alk_4 ':'Alk_4 (µmol/kg)',
                                              'SAL_real': 'SAL_ppm'})




columns = ['Cruise', 'Station', 'Sample ID', 'Time ID', 'SAL_ppm',
       'Alk_1 (µmol/kg)', 'Alk_2 (µmol/kg)', 'Alk_3 (µmol/kg)',
       'Alk_4 (µmol/kg)', 'date_measured_alk','notes']



# order columns like this: Cruise, Station, Sample ID, Time ID, Alk_1 (µmol/kg), Alk_2 (µmol/kg), Alk_3 (µmol/kg), Alk_4 (µmol/kg), Salinity (K*2), PIC (mg/L)

SS20260601_df = SS20260601_df[columns]

print(BE20260729_df.columns)
print(SS20260601_df.columns)

df = pd.concat([BE20260729_df, SS20260601_df], ignore_index=True)

In [ ]:
# only keep rows where Cruise is BE20260729 or SS20260601
df = df[df['Cruise'].isin(['BE20260729', 'SS20260601'])]

print(df['Cruise'].unique())

print(df.columns)

# normalise alkalinity measurements to a common salinity of 35.0 using the formula: Alk_mean_NTA_35 = Alk_mean * (35.0 / SAL_ppm)

columns = ['Alk_1 (µmol/kg)', 'Alk_2 (µmol/kg)', 'Alk_3 (µmol/kg)',
       'Alk_4 (µmol/kg)']

for col in columns:

    colname = col + '_NTA_35'
    df[colname] = df[col] * (35.0 / df['SAL_ppm'])

# drop columns that are not needed
df = df.drop(columns=['Alk_1 (µmol/kg)', 'Alk_2 (µmol/kg)', 'Alk_3 (µmol/kg)', 'Alk_4 (µmol/kg)'])

# average Alk_values for each row
df['Alk_mean (µmol/kg)'] = df[['Alk_1 (µmol/kg)_NTA_35', 'Alk_2 (µmol/kg)_NTA_35', 'Alk_3 (µmol/kg)_NTA_35', 'Alk_4 (µmol/kg)_NTA_35']].mean(axis=1)

# std dev for alk
df['Alk_std (µmol/kg)'] = df[['Alk_1 (µmol/kg)_NTA_35', 'Alk_2 (µmol/kg)_NTA_35', 'Alk_3 (µmol/kg)_NTA_35', 'Alk_4 (µmol/kg)_NTA_35']].std(axis=1)

# normalising alkalinity to common salinity of 35.0

df['Alk_mean_NTA_35 (µmol/kg)'] = df['Alk_mean (µmol/kg)'] * (35.0 / df['SAL_ppm'])

print(df.columns)

print(df['Sample ID'].unique())

In [ ]:
# if sample ID is AB or CD i want to create two new rows for each sample ID, one for A and one for B, and one for C and one for D, with the same values for all other columns except Sample ID which will be A or B or C or D.

df["Sample ID"] = df["Sample ID"].apply(
    lambda x: ["A", "B"] if x == "AB" else x
)

df["Sample ID"] = df["Sample ID"].apply(
    lambda x: ["C", "D"] if x == "CD" else x
)

df = df.explode("Sample ID", ignore_index=True)

df["Station"] = df["Station"].astype("Int64")

df['exp_ID']= df['Cruise'] + '_' + df['Station'].astype(str) + '_' + df['Sample ID'] 

print(df['exp_ID'].unique())

In [ ]:
# splitting in t0 and t1 based on Time ID column

df_t0 = df[df['Time ID'] == 't0']
df_t1 = df[df['Time ID'] == 't1']

# dropping all rows where there is no perfect match between t0 and t1 based on exp_ID, i.e. if there is a t0 but no t1 for the same exp_ID, drop the t0 row, and vice versa.
df_t0 = df_t0[df_t0['exp_ID'].isin(df_t1['exp_ID'])]
t0_alk_by_exp = df_t0.set_index("exp_ID")["Alk_mean_NTA_35 (µmol/kg)"]

df_t1["Alk_mean_NTA_35_diff (µmol/kg)"] = (
    df_t1["Alk_mean_NTA_35 (µmol/kg)"]
    - df_t1["exp_ID"].map(t0_alk_by_exp)
)
print(df_t1['Alk_mean_NTA_35_diff (µmol/kg)'])

df = df_t1.where(df_t1['Alk_mean_NTA_35_diff (µmol/kg)'] > 0)

print(len(df))